In [ ]:
import pandas as pd

df = pd.read_csv('Dataset.csv')

In [ ]:
df['is_autopay'] = df['PaymentMethod'].isin(['Bank transfer (automatic)','Credit card (automatic)']).astype(int)

contract_map = {
    'Month-to-month': 0,
    'One year': 1,
    'Two year': 2
}
df['contract_score'] = df['Contract'].map(contract_map)
df['monthly_charge_zscore'] = (
    (df['MonthlyCharges'] - df['MonthlyCharges'].mean()) /
    df['MonthlyCharges'].std()
)


df['has_phone_and_multiple_lines'] = (
    (df['PhoneService'] == 'Yes') & (df['MultipleLines'] == 'Yes')
).astype(int)


df['has_internet_service'] = (df['InternetService'] != 'No').astype(int)


service_cols = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]
df['total_services'] = (df[service_cols]=='Yes').sum(axis=1)


df['engagement_score'] = (
    df['has_phone_and_multiple_lines'] +
    df['has_internet_service'] +
    df['total_services']
)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(x='Churn', data=df)
plt.title("Churn Distribution")
plt.show()

sns.boxplot(x='Churn', y='MonthlyCharges', data=df)
plt.title("Monthly Charges vs Churn")
plt.show()

sns.countplot(x='Contract', hue='Churn', data=df)
plt.title("Contract Type vs Churn")
plt.xticks(rotation=15)
plt.show()

sns.histplot(data=df, x='total_services', hue='Churn', multiple='stack', bins=6)
plt.title("Total Services Used vs Churn")
plt.show()

sns.countplot(x='is_autopay', hue='Churn', data=df)
plt.title("AutoPay vs Churn")
plt.xticks([0,1], ['No AutoPay', 'AutoPay'])
plt.show()

plt.figure(figsize=(12,8))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap='coolwarm')
plt.title("Feature Correlation Heatmap")
plt.show()


In [ ]:
from sklearn.preprocessing import LabelEncoder

genderLe = LabelEncoder()
partnerLe = LabelEncoder()
dependentsLe = LabelEncoder()
phoneServiceLe = LabelEncoder()
multipleLinesLe= LabelEncoder()
internetServiceLe= LabelEncoder()
onlineSecurityLe= LabelEncoder()
onlineBackupLe= LabelEncoder()
deviceProtectionLe= LabelEncoder()
techSupportLe= LabelEncoder()
streamingTvLe= LabelEncoder()
streamingMoviesLe= LabelEncoder()
contractLe= LabelEncoder()
paperlessBillingLe= LabelEncoder()
paymentMethodLe= LabelEncoder()
churnLe = LabelEncoder()

df['gender'] = genderLe.fit_transform(df['gender'])
df['Partner'] = partnerLe.fit_transform(df['Partner'])
df['Dependents'] = dependentsLe.fit_transform(df['Dependents'])
df['PhoneService'] = phoneServiceLe.fit_transform(df['PhoneService'])
df['MultipleLines'] = multipleLinesLe.fit_transform(df['MultipleLines'])
df['InternetService'] = internetServiceLe.fit_transform(df['InternetService'])
df['OnlineSecurity'] = onlineSecurityLe.fit_transform(df['OnlineSecurity'])
df['OnlineBackup'] = onlineBackupLe.fit_transform(df['OnlineBackup'])
df['DeviceProtection'] = deviceProtectionLe.fit_transform(df['DeviceProtection'])
df['TechSupport'] = techSupportLe.fit_transform(df['TechSupport'])
df['StreamingTV'] = streamingTvLe.fit_transform(df['StreamingTV'])
df['StreamingMovies'] = streamingMoviesLe.fit_transform(df['StreamingMovies'])
df['Contract'] = contractLe.fit_transform(df['Contract'])
df['PaperlessBilling'] = paperlessBillingLe.fit_transform(df['PaperlessBilling'])
df['PaymentMethod'] = paymentMethodLe.fit_transform(df['PaymentMethod'])
df['Churn'] = churnLe.fit_transform(df['Churn'])

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)
df['TotalCharges'] = df['TotalCharges'].astype(float)

In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

X = df.drop(columns=['customerID','Churn'])
Y = df['Churn']
xtrain, xtest,ytrain,ytest = train_test_split(X,Y,test_size=0.2)

smote = SMOTE(random_state=42)
X_train_smote, Y_train_smote = smote.fit_resample(xtrain, ytrain)

In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

def evaluateModel(model, xtrain,xtest,ytrain,ytest) :
      model.fit(xtrain,ytrain)
      return model.score(xtest,ytest)   
  
kf = KFold(n_splits=5)
rf_scores =[]
xgb_scores=[]
lgm_scores=[]

for train_idx , test_idx in kf.split(X) :
    xtrain, xtest, ytrain, ytest = X_train_smote.iloc[train_idx], X_train_smote.iloc[test_idx], Y_train_smote.iloc[train_idx], Y_train_smote.iloc[test_idx]
    rf_scores.append(evaluateModel(RandomForestClassifier(),xtrain,xtest,ytrain,ytest))
    xgb_scores.append(evaluateModel(XGBClassifier(),xtrain,xtest,ytrain,ytest))
    lgm_scores.append(evaluateModel(LGBMClassifier(),xtrain,xtest,ytrain,ytest))
    
print("\nRandom Forest : ", rf_scores )
print("\nXGBoost : ", xgb_scores )
print("\nLightgbm : ", lgm_scores )

# Random Forest :  [0.7806955287437899, 0.7906316536550745, 0.7920511000709723, 0.7876420454545454, 0.6193181818181818]
# XGBoost :  [0.7785663591199432, 0.7778566359119943, 0.7906316536550745, 0.7741477272727273, 0.6505681818181818]
# Lightgbm :  [0.7792760823278921, 0.7934705464868701, 0.7877927608232789, 0.7862215909090909, 0.6079545454545454]

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
import matplotlib.pyplot as plt


X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

# Apply SMOTE to only the training data
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Best parameters found
best_params = {
    'n_estimators': 200,
    'max_depth': 20,
    'min_samples_split': 5,
    'min_samples_leaf': 2,
    'max_features': 'log2'
}


final_rf = RandomForestClassifier(**best_params, random_state=42)
final_rf.fit(X_train_smote, y_train_smote)


y_pred = final_rf.predict(X_test)

# Evaluate
print("✅ Accuracy:", accuracy_score(y_test, y_pred))
print("\n📊 Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\n📝 Classification Report:\n", classification_report(y_test, y_pred))

# ROC-AUC
y_proba = final_rf.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_proba)
print(f"\n🔥 ROC AUC Score: {roc_auc:.4f}")


# Accuracy: 0.7629524485450674
# 📊 Confusion Matrix:
#  [[846 189]
#  [145 229]]

# 📝 Classification Report:
#                precision    recall  f1-score   support

#            0       0.85      0.82      0.84      1035
#            1       0.55      0.61      0.58       374

#     accuracy                           0.76      1409
#    macro avg       0.70      0.71      0.71      1409
# weighted avg       0.77      0.76      0.77      1409


# 🔥 ROC AUC Score: 0.8298


In [ ]:
# Use the best parameters for training the final model
rf_best = RandomForestClassifier(
    class_weight='balanced',
    max_depth=20,
    min_samples_leaf=1,
    min_samples_split=5,
    n_estimators=200,
    random_state=42
)


rf_best.fit(X_train_smote, y_train_smote)


y_pred = rf_best.predict(X_test)

# Evaluation metrics 
print("✅ Accuracy:", accuracy_score(y_test, y_pred))
print("📊 Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\n📝 Classification Report:")
print(classification_report(y_test, y_pred))

# ROC AUC Score
roc_auc = roc_auc_score(y_test, rf_best.predict_proba(X_test)[:, 1])
print(f"\n🔥 ROC AUC Score: {roc_auc:.4f}")

# ✅ Accuracy: 0.7707594038325053
# 📊 Confusion Matrix:
# [[864 171]
#  [152 222]]

# 📝 Classification Report:
#               precision    recall  f1-score   support

#            0       0.85      0.83      0.84      1035
#            1       0.56      0.59      0.58       374

#     accuracy                           0.77      1409
#    macro avg       0.71      0.71      0.71      1409
# weighted avg       0.77      0.77      0.77      1409


# 🔥 ROC AUC Score: 0.8286


In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Split your data (assuming you already have X and y defined)
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Define base models
base_learners = [
    ('rf', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)),
    ('xgb', XGBClassifier(scale_pos_weight=1, random_state=42)),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
]

# Stacked model with final estimator as Logistic Regression
stacked_model = StackingClassifier(
    estimators=base_learners, 
   final_estimator=RandomForestClassifier(n_estimators=100, random_state=42)
)

# Train the stacked model
stacked_model.fit(X_train, y_train)

# Make predictions
y_pred = stacked_model.predict(X_test)
y_pred_prob = stacked_model.predict_proba(X_test)[:, 1]

# Evaluation metrics
print("✅ Accuracy:", accuracy_score(y_test, y_pred))
print("📊 Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\n📝 Classification Report:")
print(classification_report(y_test, y_pred))

# ROC AUC Score
roc_auc = roc_auc_score(y_test, y_pred_prob)
print(f"\n🔥 ROC AUC Score: {roc_auc}")

### Stacking Model Comparison:

# | **Final Estimator**    | **Accuracy** | **Precision (Churn = 1)** | **Recall (Churn = 1)** | **F1-Score (Churn = 1)** | **ROC AUC** | **Notes**                                  |
# |------------------------|--------------|---------------------------|------------------------|--------------------------|-------------|--------------------------------------------|
# | **Logistic Regression** | 76.29%       | 0.55                      | 0.61                   | 0.58                     | 0.83        | Lower precision for churn, decent recall.  |
# | **Random Forest**       | 81.26%       | 0.68                      | 0.55                   | 0.61                     | 0.86        | Higher accuracy, better for balancing classes. |
# | **XGBClassifier**       | 80.98%       | 0.67                      | 0.55                   | 0.61                     | 0.84        | Similar to Random Forest, with slightly lower ROC AUC. |



In [ ]:
import joblib

joblib.dump(stacked_model,"TelecomCustomerChurnPrediction.pkl")